In [ ]:
!pip install lyricsgenius

In [ ]:
pip install pandas lyricsgenius nltk better-profanity langdetect

In [ ]:
import random
import re
import lyricsgenius
import pandas as pd

token = "MM3rhwjFkBZVZU5k7fixqT4u_rCu9u7kPnLCEWZdKiEoxm0ggcRLFvIwbgg98cEL"
genius = lyricsgenius.Genius(token, timeout=30)

genres = ['pop', 'rock', 'rap', 'r-b', 'country', 'indie', 'blues', 'metal', 'folk']
songs_per_genre = 100

def get_song_metadata(title: str, artist_name: str) -> dict:
    search_term = f"{title} {artist_name}".strip()
    try:
        res = genius.search_songs(search_term, per_page=1, page=1)
        hits = res.get('hits', [])
        if hits:
            return hits[0].get('result', {})
    except Exception as e:
        print(f"Metadata search failed for '{title}' by {artist_name}: {e}")
    return {}

all_songs = []

for genre in genres:
    print(f"מושך שירים מז'אנר: {genre}")
    page = 1
    candidate_hits = []

    while page and len(candidate_hits) < songs_per_genre * 2:
        try:
            res = genius.tag(genre, page=page)
        except Exception as e:
            print(f"Failed to fetch tag page for {genre}: {e}")
            break

        if not res or 'hits' not in res:
            break
        candidate_hits.extend(res['hits'])
        page = res.get('next_page')

    if not candidate_hits:
        print(f"No hits found for {genre}.")
        continue

    unique_hits = {hit['url']: hit for hit in candidate_hits}
    selected_hits = random.sample(
        list(unique_hits.values()),
        min(songs_per_genre, len(unique_hits))
    )

    for hit in selected_hits:
        artist_name = hit['artists'][0] if hit['artists'] else ""
        title = hit['title']

        song_info = get_song_metadata(title, artist_name)
        release_year = None
        if song_info.get('release_date_components'):
            release_year = song_info['release_date_components'].get('year')

        lyrics = ""
        try:
            lyrics = genius.lyrics(song_url=hit['url']) or ""
        except Exception as e:
            print(f"Lyrics failed for '{title}': {e}")

        all_songs.append({
            'song_title': title,
            'artist_name': artist_name,
            'release_year': int(release_year) if release_year else None,
            'genre': genre,
            'lyrics': lyrics,
            'pageviews': song_info.get('stats', {}).get('pageviews', 0),
            'featured_artists_count': len(hit['featured_artists']),
            'primary_tag': genre,
            'language': song_info.get('language'),
            'source_url': hit['url']
        })

raw_df = pd.DataFrame(all_songs)
raw_df.to_csv('raw_genius_songs.csv', index=False, encoding='utf-8-sig')
print("הסתיים שלב המשיכה! הקובץ הגולמי נשמר בהצלחה.")

In [ ]:
import re
from collections import Counter
import pandas as pd
import nltk
from nltk.corpus import stopwords
from better_profanity import profanity

# --- תיקון השגיאה והורדת משאבי ה-NLP ---
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # <-- השורה שפתרה את ה-LookupError

STOP_WORDS = set(stopwords.words('english'))

def analyze_lyrics_advanced(lyrics):
    if not lyrics or not isinstance(lyrics, str):
        return {k: 0 for k in ['total_word_count', 'count_chorus', 'count_verse', 'count_intro', 'count_bridge', 'unique_special_word', 'special_words_ratio', 'special_word_without_chorus', 'unique_ratio_no_repeated_chorus', 'profanity_count', 'profanity_ratio', 'adjectives_count', 'adjectives_ratio']}
    
    # 1. ספירת חלקי השיר
    raw_tags = re.findall(r'\[(.*?)\]', lyrics)
    chorus_count = sum(1 for t in raw_tags if 'chorus' in t.lower())
    verse_count = sum(1 for t in raw_tags if 'verse' in t.lower())
    intro_count = sum(1 for t in raw_tags if 'intro' in t.lower())
    bridge_count = sum(1 for t in raw_tags if 'bridge' in t.lower())
    
    # 2. הסרת פזמונים חוזרים
    blocks = re.split(r'(\[.*?\])', lyrics)
    modified_blocks = []
    chorus_counter = 0
    skip_current_block = False
    
    for block in blocks:
        if block.startswith('[') and block.endswith(']'):
            if 'chorus' in block.lower():
                chorus_counter += 1
                if chorus_counter > 1:
                    skip_current_block = True
                    continue
            skip_current_block = False
            modified_blocks.append(block)
        else:
            if skip_current_block:
                continue
            modified_blocks.append(block)
            
    lyrics_no_repeated_chorus = "".join(modified_blocks)
    
    def get_clean_words(text):
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'[^\w\s]', '', text).lower()
        return text.split()
    
    words_all = get_clean_words(lyrics)
    words_no_repeated_chorus = get_clean_words(lyrics_no_repeated_chorus)
    
    total_words_count = len(words_all)
    if total_words_count == 0:
        return {k: 0 for k in ['total_word_count', 'count_chorus', 'count_verse', 'count_intro', 'count_bridge', 'unique_special_word', 'special_words_ratio', 'special_word_without_chorus', 'unique_ratio_no_repeated_chorus', 'profanity_count', 'profanity_ratio', 'adjectives_count', 'adjectives_ratio']}
        
    special_words = [w for w in words_all if w not in STOP_WORDS]
    special_words_no_chorus = [w for w in words_no_repeated_chorus if w not in STOP_WORDS]
    
    # 3. ספירת קללות ומילות תואר
    profanity_count = sum(1 for w in special_words if profanity.contains_profanity(w))
    
    tokens = nltk.word_tokenize(re.sub(r'\[.*?\]', '', lyrics))
    tagged_words = nltk.pos_tag(tokens)
    adjectives_count = sum(1 for word, tag in tagged_words if tag in ('JJ', 'JJR', 'JJS'))
    
    # 4. חישוב מדדים ויחסים
    unique_special_word = len(set(special_words))
    total_special_words = len(special_words)
    special_word_without_chorus = len(special_words_no_chorus)
    
    special_words_ratio = unique_special_word / total_special_words if total_special_words > 0 else 0
    unique_ratio_no_repeated_chorus = unique_special_word / special_word_without_chorus if special_word_without_chorus > 0 else 0
    profanity_ratio = profanity_count / total_words_count
    adjectives_ratio = adjectives_count / total_words_count
    
    return {
        'total_word_count': total_words_count,
        'count_chorus': chorus_count,
        'count_verse': verse_count,
        'count_intro': intro_count,
        'count_bridge': bridge_count,
        'unique_special_word': unique_special_word,
        'special_words_ratio': special_words_ratio,
        'special_word_without_chorus': special_word_without_chorus,
        'unique_ratio_no_repeated_chorus': unique_ratio_no_repeated_chorus,
        'profanity_count': profanity_count,
        'profanity_ratio': profanity_ratio,
        'adjectives_count': adjectives_count,
        'adjectives_ratio': adjectives_ratio
    }

# --- טעינת הקובץ הגולמי ועיבודו ---
print("טוען קובץ גולמי ומעבד נתונים...")
df = pd.read_csv('raw_genius_songs.csv')

# הרצת הניתוח על כל שורה בנפרד והפיכת התוצאה לעמודות חדשות
nlp_results = df['lyrics'].apply(analyze_lyrics_advanced)
nlp_df = pd.DataFrame(list(nlp_results))

# איחוד הטבלה המקורית עם העמודות החדשות
final_df = pd.concat([df, nlp_df], axis=1)

# שמירת הדאטה-סט הסופי והמועשר
final_df.to_csv('final_genius_enriched_songs.csv', index=False, encoding='utf-8-sig')

print("הסתיים שלב ה-NLP! הקובץ הסופי מוכן:")
print(final_df.info())